<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/analizar_movimiento_MediaPipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Instalar dependencias

In [ ]:
!pip install -q mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 7.8 MB/s eta 0:00:00


## 2. Subir el video

Ejecuta la celda y selecciona tu archivo de video (mp4, mov, etc). También puedes montar Google Drive si prefieres usar un video ya guardado ahí.

In [ ]:
from google.colab import files

subido = files.upload()
ruta_video = list(subido.keys())[0]
print(f"Video cargado: {ruta_video}")

Saving ejercicio1.mp4 to ejercicio1.mp4
Video cargado: ejercicio1.mp4


In [ ]:
import os
import cv2
import math
import urllib.request
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from google.colab import files
from IPython.display import Video, display

In [ ]:
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

In [ ]:
BaseOptions = python.BaseOptions
MODEL_PATH = 'pose_landmarker.task'

options = vision.PoseLandmarkerOptions(

    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),

    running_mode=vision.RunningMode.VIDEO,

    num_poses=1,

    min_pose_detection_confidence=0.3,

    min_pose_presence_confidence=0.3,

    min_tracking_confidence=0.3
)

landmarker = vision.PoseLandmarker.create_from_options(
    options
)

print("Detector creado correctamente.")

Detector creado correctamente.


In [ ]:
def distancia(p1, p2):

    return math.sqrt(

        (p1.x - p2.x) ** 2 +

        (p1.y - p2.y) ** 2 +

        (p1.z - p2.z) ** 2

    )

In [ ]:
def calcular_angulo(a, b, c):

    ba = np.array([

        a.x - b.x,

        a.y - b.y

    ])

    bc = np.array([

        c.x - b.x,

        c.y - b.y

    ])

    producto = np.dot(ba, bc)

    norma = (

        np.linalg.norm(ba) *

        np.linalg.norm(bc)

    )

    if norma == 0:

        return 0

    coseno = producto / norma

    coseno = np.clip(
        coseno,
        -1.0,
        1.0
    )

    angulo = np.degrees(
        np.arccos(coseno)
    )

    return angulo

In [ ]:
def detectar_movimiento(
    landmarks,
    historial
):

    # PUNTOS DEL CUERPO

    nariz = landmarks[0]

    hombro_izq = landmarks[11]

    hombro_der = landmarks[12]

    codo_izq = landmarks[13]

    codo_der = landmarks[14]

    muneca_izq = landmarks[15]

    muneca_der = landmarks[16]

    cadera_izq = landmarks[23]

    cadera_der = landmarks[24]

    rodilla_izq = landmarks[25]

    rodilla_der = landmarks[26]

    tobillo_izq = landmarks[27]

    tobillo_der = landmarks[28]


    # BRAZOS ARRIBA

    if (

        muneca_izq.y < hombro_izq.y and

        muneca_der.y < hombro_der.y

    ):

        return "BRAZOS ARRIBA"


    # BRAZO IZQUIERDO ARRIBA

    if (

        muneca_izq.y < hombro_izq.y

    ):

        return "BRAZO IZQUIERDO ARRIBA"


    # BRAZO DERECHO ARRIBA

    if (

        muneca_der.y < hombro_der.y

    ):

        return "BRAZO DERECHO ARRIBA"


    # ÁNGULOS DE LAS RODILLAS

    angulo_izquierdo = calcular_angulo(

        cadera_izq,

        rodilla_izq,

        tobillo_izq

    )


    angulo_derecho = calcular_angulo(

        cadera_der,

        rodilla_der,

        tobillo_der

    )


    # SENTADO


    if (

        angulo_izquierdo < 120 and

        angulo_derecho < 120

    ):

        return "SENTADO"


    # AGACHADO


    altura_cadera = (

        cadera_izq.y +

        cadera_der.y

    ) / 2


    altura_rodillas = (

        rodilla_izq.y +

        rodilla_der.y

    ) / 2


    if altura_cadera > altura_rodillas:

        return "AGACHADO"


    # DETECCIÓN DE DESPLAZAMIENTO

    posicion_actual = (

        (

            tobillo_izq.x +

            tobillo_der.x

        ) / 2,

        (

            tobillo_izq.y +

            tobillo_der.y

        ) / 2

    )


    historial.append(
        posicion_actual
    )


    if len(historial) > 10:

        historial.pop(0)


        desplazamiento = math.sqrt(

            (

                historial[-1][0] -

                historial[0][0]

            ) ** 2

            +

            (

                historial[-1][1] -

                historial[0][1]

            ) ** 2

        )


        if desplazamiento > 0.03:

            return "CAMINANDO"


    # ESTADO POR DEFECTO

    return "DE PIE"

In [ ]:
# ============================================
# ABRIR VIDEO
# ============================================

cap = cv2.VideoCapture(
    ruta_video
)


# ============================================
# INFORMACIÓN DEL VIDEO
# ============================================

fps = cap.get(
    cv2.CAP_PROP_FPS
)


if fps == 0:

    fps = 30


width = int(

    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )

)


height = int(

    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )

)


total_frames = int(

    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )

)


print("FPS:", fps)

print("Ancho:", width)

print("Alto:", height)

print("Frames:", total_frames)


# ============================================
# VIDEO DE SALIDA
# ============================================

VIDEO_OUTPUT = (
    "video_movimientos_detectados.mp4"
)


fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)


out = cv2.VideoWriter(

    VIDEO_OUTPUT,

    fourcc,

    fps,

    (width, height)

)


# ============================================
# HISTORIAL
# ============================================

historial = []


frame_number = 0


# ============================================
# PROCESAR VIDEO
# ============================================

while True:


    ret, frame = cap.read()


    if not ret:

        break


    # Convertir de BGR a RGB

    rgb = cv2.cvtColor(

        frame,

        cv2.COLOR_BGR2RGB

    )


    # Crear imagen compatible con MediaPipe

    mp_image = mp.Image(

        image_format=mp.ImageFormat.SRGB,

        data=rgb

    )


    # Timestamp en milisegundos

    timestamp_ms = int(

        frame_number *

        1000 /

        fps

    )


    # Detectar pose

    result = landmarker.detect_for_video(

        mp_image,

        timestamp_ms

    )


    movimiento = "NO DETECTADO"


    # ========================================
    # SI ENCUENTRA UNA PERSONA
    # ========================================

    if len(result.pose_landmarks) > 0:


        landmarks = result.pose_landmarks[0]


        # Detectar movimiento

        movimiento = detectar_movimiento(

            landmarks,

            historial

        )


        # ====================================
        # CONEXIONES DEL CUERPO
        # ====================================

        conexiones = [

            (11, 12),


            # Brazo izquierdo

            (11, 13),

            (13, 15),


            # Brazo derecho

            (12, 14),

            (14, 16),


            # Tronco

            (11, 23),

            (12, 24),

            (23, 24),


            # Pierna izquierda

            (23, 25),

            (25, 27),


            # Pierna derecha

            (24, 26),

            (26, 28)

        ]


        # ====================================
        # DIBUJAR PUNTOS
        # ====================================

        for punto in landmarks:


            x = int(

                punto.x *

                width

            )


            y = int(

                punto.y *

                height

            )


            if (

                0 <= x < width and

                0 <= y < height

            ):


                cv2.circle(

                    frame,

                    (x, y),

                    5,

                    (0, 255, 0),

                    -1

                )


        # ====================================
        # DIBUJAR CONEXIONES
        # ====================================

        for p1, p2 in conexiones:


            x1 = int(

                landmarks[p1].x *

                width

            )


            y1 = int(

                landmarks[p1].y *

                height

            )


            x2 = int(

                landmarks[p2].x *

                width

            )


            y2 = int(

                landmarks[p2].y *

                height

            )


            cv2.line(

                frame,

                (x1, y1),

                (x2, y2),

                (255, 255, 0),

                3

            )


    # ========================================
    # PANEL DE INFORMACIÓN
    # ========================================

    overlay = frame.copy()


    cv2.rectangle(

        overlay,

        (20, 20),

        (650, 125),

        (0, 0, 0),

        -1

    )


    frame = cv2.addWeighted(

        overlay,

        0.70,

        frame,

        0.30,

        0

    )


    # Título

    cv2.putText(

        frame,

        "MEDIA PIPE - ANALISIS CORPORAL",

        (40, 55),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.75,

        (255, 255, 255),

        2

    )


    # Movimiento detectado

    cv2.putText(

        frame,

        "MOVIMIENTO: " + movimiento,

        (40, 100),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.75,

        (0, 255, 255),

        2

    )


    # ========================================
    # GUARDAR FRAME
    # ========================================

    out.write(frame)


    frame_number += 1


    # Mostrar progreso

    if frame_number % 100 == 0:

        print(

            "Procesados:",

            frame_number,

            "/",

            total_frames

        )


# ============================================
# CERRAR
# ============================================

cap.release()

out.release()

landmarker.close()


print()
print("===================================")
print("PROCESAMIENTO TERMINADO")
print("===================================")
print(
    "Archivo:",
    VIDEO_OUTPUT
)

FPS: 29.97
Ancho: 640
Alto: 360
Frames: 10952
Procesados: 100 / 10952
Procesados: 200 / 10952
Procesados: 300 / 10952
Procesados: 400 / 10952
Procesados: 500 / 10952
Procesados: 600 / 10952
Procesados: 700 / 10952
Procesados: 800 / 10952
Procesados: 900 / 10952
Procesados: 1000 / 10952
Procesados: 1100 / 10952
Procesados: 1200 / 10952
Procesados: 1300 / 10952
Procesados: 1400 / 10952
Procesados: 1500 / 10952
Procesados: 1600 / 10952
Procesados: 1700 / 10952
Procesados: 1800 / 10952
Procesados: 1900 / 10952
Procesados: 2000 / 10952
Procesados: 2100 / 10952
Procesados: 2200 / 10952
Procesados: 2300 / 10952
Procesados: 2400 / 10952
Procesados: 2500 / 10952
Procesados: 2600 / 10952
Procesados: 2700 / 10952
Procesados: 2800 / 10952
Procesados: 2900 / 10952
Procesados: 3000 / 10952
Procesados: 3100 / 10952
Procesados: 3200 / 10952
Procesados: 3300 / 10952
Procesados: 3400 / 10952
Procesados: 3500 / 10952
Procesados: 3600 / 10952
Procesados: 3700 / 10952
Procesados: 3800 / 10952
Procesados: 3

In [ ]:
files.download(
    VIDEO_OUTPUT
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>